#### Question Statement

## Q.4 Writing Viterbi Algorithm for the Primer
Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.


Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

## Hidden Markov Models (HMMs)
A compact way to model sequences where you can’t see the true “state” but do observe outputs (like DNA bases).

States (hidden): e.g. Exon (E), Intron (I), Splice‑site (S)

Observations: the actual nucleotides (A, C, G, T)

Transitions: chances of going E→E, E→S, S→I, I→I, etc.

Emissions: probability of each base given the state, e.g. P(A | E), P(G | I)

Start probabilities: likelihood of beginning in E, I, or S

Together, these let you infer the most likely hidden path behind a DNA sequence.

In [1]:
import numpy as np
import math

In [2]:
bases = ['A', 'C', 'G', 'T']
# our hidden states: exon, splice site, intron
hidden_states = ['E', 'S', 'I']

# starting probabilities for each hidden state
start_probabilities = {'E': 1.0, 'S': 0.0, 'I': 0.0}

# how likely we jump from one state to another (including Start and End)
transition_probabilities = {
    'Start': {'E': 1.0, 'S': 0.0, 'I': 0.0, 'End': 0.0},
    'E':     {'E': 0.9, 'S': 0.1, 'I': 0.0, 'End': 0.0},
    'S':     {'E': 0.0, 'S': 0.0, 'I': 1.0, 'End': 0.0},
    'I':     {'E': 0.0, 'S': 0.0, 'I': 0.9, 'End': 0.1}
}

# probability of seeing each base when in a given state
emission_probabilities = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    'S': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

def safe_log(x: float) -> float:
    return -math.inf if x == 0 else math.log(x)

def compute_log_probability(state_path: str, observations: str) -> float:
    if len(state_path) != len(observations):
        raise ValueError("State path & observation sequence must be of same len")

    log_prob = 0.0
    previous_state = 'Start'

    for state, obs in zip(state_path, observations):
        log_prob += safe_log(transition_probabilities[previous_state][state])
        log_prob += safe_log(emission_probabilities[state][obs])
        previous_state = state

    # account for the transition from the last state to End
    if previous_state == 'I':
        log_prob += safe_log(transition_probabilities[previous_state]['End'])

    return log_prob

In [3]:
state_sequence = "EEEEEEEEEEEEEEEEEESIIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
log_probability = compute_log_probability(state_sequence, observed_sequence)
print(log_probability)

-41.21967768602254


## The Viterbi Algorithm
A dynamic‑programming method to uncover the most probable path of hidden states behind an observed sequence:

#### Initialization
Fill the first column of your score matrix using the start‑state probabilities combined with the emission probability for the first observation.

#### Recursion
Move along the sequence one position at a time. For each state, pick the predecessor that maximizes the cumulative score, add the transition and emission log‑probabilities, and record backpointers.

#### Termination
Identify which state at the final position has the highest score.

#### Backtracking
Trace back through your stored pointers to reconstruct the sequence of states that led to that maximum score.

In [4]:
def viterbi_algorithm(observed_seq):
    n_states = len(hidden_states)
    seq_length = len(observed_seq)

    # v_table[state_index, pos] = best log-prob up to pos ending in that state
    v_table = np.full((n_states, seq_length), -np.inf)
    # back_ptr[state_index, pos] = which state_index led here
    back_ptr = np.zeros((n_states, seq_length), dtype=int)

    # --- Initialization: at the first position, combine start + emission ---
    first_obs = observed_seq[0]
    for s_idx, state in enumerate(hidden_states):
        start_log = safe_log(start_probabilities[state])
        emit_log = safe_log(emission_probabilities[state][first_obs])
        v_table[s_idx, 0] = start_log + emit_log

    # --- Recursion: fill in v_table for each subsequent position ---
    for pos in range(1, seq_length):
        obs = observed_seq[pos]
        for curr_idx, curr_state in enumerate(hidden_states):
            best_prev_log = -np.inf
            best_prev_idx = 0
            for prev_idx, prev_state in enumerate(hidden_states):
                trans_log = safe_log(transition_probabilities[prev_state][curr_state])
                candidate = v_table[prev_idx, pos - 1] + trans_log
                if candidate > best_prev_log:
                    best_prev_log = candidate
                    best_prev_idx = prev_idx
            emit_log = safe_log(emission_probabilities[curr_state][obs])
            v_table[curr_idx, pos] = best_prev_log + emit_log
            back_ptr[curr_idx, pos] = best_prev_idx

    # picking the state with highest log-prob in the final column
    last_idx = int(np.argmax(v_table[:, -1]))
    best_path_indices = [last_idx]

    # follow back pointers from end to start
    for pos in range(seq_length - 1, 0, -1):
        last_idx = back_ptr[last_idx, pos]
        best_path_indices.insert(0, last_idx)

    # convert index path back to state labels
    best_states = ''.join(hidden_states[i] for i in best_path_indices)
    best_log_prob = float(np.max(v_table[:, -1]))

    return best_states, best_log_prob

In [5]:
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# runnig the Viterbi algorithm
best_path, best_log_prob = viterbi_algorithm(observed_sequence)

print(f"Most likely state sequence: {best_path}")
print(f"Log probability of that path: {best_log_prob}")

Most likely state sequence: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of that path: -38.677666280562796
